# K-Nearest Neighbors - Heart Disease

Explores a tuned KNN classifier on the cleaned splits.

**Method.** Hyperparameters are tuned with stratified 5-fold `GridSearchCV` on the
**training** set; the tuned model is evaluated on the **validation** set. The
**test** set is deliberately left untouched here - it is reserved for the final
cross-model comparison notebook, so the hold-out is spent only once.

KNN needs scaled features (it is distance-based); scaling is applied to the
continuous columns only, inside the pipeline, via `dataset.build_scaler`.

In [ ]:
import sys
from pathlib import Path

# Make the scripts/ helpers importable from notebooks/.
sys.path.insert(0, str(Path.cwd().parent / "scripts"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import GridSearchCV, cross_val_score
from sklearn.metrics import (
    roc_auc_score, recall_score, precision_score, f1_score, accuracy_score,
    confusion_matrix, roc_curve, ConfusionMatrixDisplay,
)

from dataset import load_splits, get_xy
from train_models import build_pipeline, MODELS, CV

MODEL_NAME = "knn"

## Load data

In [ ]:
train, val, test = load_splits()
X_train, y_train = get_xy(train)
X_val, y_val = get_xy(val)

print(f"train {X_train.shape}  val {X_val.shape}  (test held back)")
print(f"train disease rate {y_train.mean():.3f}  val {y_val.mean():.3f}")

## Tune hyperparameters (train, 5-fold CV)

The estimator and grid come straight from the `MODELS` registry in
`train_models.py`, so the notebook and the batch script stay in sync.

In [ ]:
cfg = MODELS[MODEL_NAME]
pipe = build_pipeline(cfg["estimator"], cfg["needs_scaling"], list(X_train.columns))
search = GridSearchCV(pipe, cfg["grid"], cv=CV, scoring="roc_auc", n_jobs=-1)
search.fit(X_train, y_train)

best = search.best_estimator_
print("best params:", search.best_params_)
print(f"cv roc-auc: {search.best_score_:.3f}")

## Validation metrics

In [ ]:
proba = best.predict_proba(X_val)[:, 1]
pred = best.predict(X_val)

metrics = {
    "roc_auc": roc_auc_score(y_val, proba),
    "recall": recall_score(y_val, pred),
    "precision": precision_score(y_val, pred),
    "f1": f1_score(y_val, pred),
    "accuracy": accuracy_score(y_val, pred),
}
pd.Series(metrics, name=MODEL_NAME).round(3)

In [ ]:
ConfusionMatrixDisplay(
    confusion_matrix(y_val, pred), display_labels=["no disease", "disease"]
).plot(cmap="Blues", colorbar=False)
plt.title(f"{MODEL_NAME} - validation confusion matrix")
plt.show()

## How performance varies with `k`

Holding the best `weights`/`metric` fixed, sweep `n_neighbors` and plot the
cross-validated ROC-AUC on the training set. This shows the bias/variance
trade-off: very small `k` overfits, very large `k` oversmooths.

In [ ]:
ks = list(range(1, 31, 2))
scores = []
for k in ks:
    params = {
        "model__n_neighbors": k,
        "model__weights": search.best_params_["model__weights"],
        "model__metric": search.best_params_["model__metric"],
    }
    trial = build_pipeline(cfg["estimator"], cfg["needs_scaling"], list(X_train.columns))
    trial.set_params(**params)
    scores.append(cross_val_score(trial, X_train, y_train, cv=CV, scoring="roc_auc").mean())

plt.plot(ks, scores, marker="o")
plt.axvline(search.best_params_["model__n_neighbors"], color="gray", ls="--",
            label=f"chosen k={search.best_params_['model__n_neighbors']}")
plt.xlabel("n_neighbors (k)")
plt.ylabel("CV ROC-AUC")
plt.title("KNN - ROC-AUC vs k")
plt.legend()
plt.show()

## ROC curve (validation)

In [ ]:
fpr, tpr, _ = roc_curve(y_val, proba)
plt.plot(fpr, tpr, label=f"{MODEL_NAME} (AUC={metrics['roc_auc']:.3f})")
plt.plot([0, 1], [0, 1], ls="--", color="gray")
plt.xlabel("False positive rate")
plt.ylabel("True positive rate")
plt.title("KNN - validation ROC curve")
plt.legend()
plt.show()

## Notes

- Test set intentionally not touched - see the comparison notebook for the
  single final test evaluation of the selected model.
- To explore a different model, copy this notebook and change `MODEL_NAME` to
  any key in `MODELS` (`logreg`, `svm`, `decision_tree`, `random_forest`,
  `gradient_boosting`, `hist_gradient_boosting`, `naive_bayes`). The `k`-curve
  cell is KNN-specific and would be swapped for a model-appropriate diagnostic.